# HSH Amends — verify first, then look

This notebook proves our central claim **before** it shows you a single row.

It fetches the sample from one host and the trust anchor from a **different** host.
Those two would have to collude to fool you. That is the design: where you
*fetch* and where you *anchor trust* are separate questions.

Run the cells in order. By cell 5 you will have verified an ed25519 signature
and recomputed all 5,645 record hashes yourself. Nothing here calls an HSH API,
and no HSH code runs — only Python's standard library and `openssl`.
---

**One thing to be straight about.** This notebook runs the checks for
you, and it is also the thing telling you they passed. If you want
certainty rather than convenience, run the four commands yourself —
they are short, they are in every cell below, and `verify_sample.py`
in the sample does the same thing in one command.

No arrangement of hosts fixes that. Putting this notebook somewhere
else would not help, because it would still be the thing reporting its
own result. What the two hosts **do** protect is the data: an altered
sample will not match a key published independently of it, and you
will catch it. That is the claim, and it holds.

## 0 · Where things come from

Two different hosts, deliberately.

In [ ]:
# ── THE TWO HOSTS ────────────────────────────────────────────────────
# The dataset host. Public, anonymous, no account needed.
# Treat it as UNTRUSTED — that is the whole point of the next cell.
SAMPLE_BASE = "https://huggingface.co/datasets/HSH-Intelligence/hsh-amends-sample/resolve/main"

# The trust anchor. A DIFFERENT host, append-only, independent of the
# one above. Do not change this to a mirror on the dataset host: the
# separation is what makes the check mean anything.
ANCHOR_BASE = "https://raw.githubusercontent.com/hshintelligence/hsh-transparency/main/amends"
# ─────────────────────────────────────────────────────────────────────

SAMPLE_FILES = [
    "hsh-amends-sample.csv",
    "hsh-amends-sample.parquet",
    "MANIFEST.txt",
    "MANIFEST.txt.sig",
    "COVERAGE-ATTESTATION.json",
    "COVERAGE-ATTESTATION.json.sig",
    # The directory seal, the composition and the script itself.
    # Without these the notebook cannot run the verifier that ships
    # with the sample, and would be redoing part of its work by hand.
    "SHA256SUMS",
    "SHA256SUMS.sig",
    "SAMPLE-COMPOSITION.json",
    "verify_sample.py",
    "README.md",
]
print('sample :', SAMPLE_BASE)
print('anchor :', ANCHOR_BASE)

## 1 · Fetch the sample

From the dataset host. Treat this host as untrusted — that is the point.

In [ ]:
import hashlib, json, os, subprocess, urllib.request

os.makedirs('sample', exist_ok=True)
for name in SAMPLE_FILES:
    urllib.request.urlretrieve(f'{SAMPLE_BASE}/{name}', f'sample/{name}')
    print(f'{os.path.getsize(f"sample/{name}"):>12,}  {name}')

## 2 · Fetch the trust anchor from a **different** host

The public key and the fingerprint come from the transparency repository,
not from whoever served you the data. If the dataset host altered the files
*and* the key, it still cannot alter this.

In [ ]:
for name in ['HSH-SIGNING-KEY.pub', 'KEY-FINGERPRINT.txt', 'TRUST-ROOTS.json']:
    urllib.request.urlretrieve(f'{ANCHOR_BASE}/{name}', name)

der = subprocess.run(['openssl','pkey','-pubin','-in','HSH-SIGNING-KEY.pub',
                      '-outform','DER'], capture_output=True).stdout
got = hashlib.sha256(der).hexdigest()
want = open('KEY-FINGERPRINT.txt').read().split()[0]
print('key fingerprint :', got)
print('published       :', want)
assert got == want, 'KEY FINGERPRINT MISMATCH — stop here'
print('\nthe key matches the one published independently.')

## 3 · Verify the signature

One `openssl` command. If the manifest was altered by so much as one byte,
this fails.

In [ ]:
r = subprocess.run(['openssl','pkeyutl','-verify','-pubin',
                    '-inkey','HSH-SIGNING-KEY.pub','-rawin',
                    '-in','sample/MANIFEST.txt',
                    '-sigfile','sample/MANIFEST.txt.sig'],
                   capture_output=True)
print((r.stdout + r.stderr).decode().strip())
assert r.returncode == 0, 'SIGNATURE DID NOT VERIFY'

# the coverage attestation is signed by the same key
r2 = subprocess.run(['openssl','pkeyutl','-verify','-pubin',
                     '-inkey','HSH-SIGNING-KEY.pub','-rawin',
                     '-in','sample/COVERAGE-ATTESTATION.json',
                     '-sigfile','sample/COVERAGE-ATTESTATION.json.sig'],
                    capture_output=True)
print('attestation    :', (r2.stdout + r2.stderr).decode().strip())
assert r2.returncode == 0

## 4 · Recompute every record hash yourself

This is the claim that matters, and it is **byte-identical** to the check
that applies to the full 15,326,990-row product. Nothing about it is
special to a sample.

In [ ]:
import csv

claimed = {}
for line in open('sample/MANIFEST.txt'):
    if line.startswith('#'):
        continue
    p = line.split()
    if len(p) == 2:
        claimed[p[0]] = p[1]

n = bad = 0
with open('sample/hsh-amends-sample.csv', newline='', encoding='utf-8') as fh:
    for row in csv.DictReader(fh):
        n += 1
        h = hashlib.sha256(row['payload'].encode()).hexdigest()
        if h != row['payload_sha'] or claimed.get(row['record_uid']) != h:
            bad += 1

print(f'{n:,} records recomputed, {bad} mismatch(es)')
assert bad == 0
print('\nEvery row you hold hashes to the value a signed manifest claims.')
print('You have now proven that without trusting either host.')

## 4b · Now run the verifier we ship, which does all five

Everything above is done by hand so you can read each step. This cell
runs `verify_sample.py` — the script that travels in the sample directory
— against the files you just downloaded, with the fingerprint from the
anchor.

It is here because the cells above cover three of its five checks. They
do not compare the files against the manifest header, and they do not
verify `SHA256SUMS`. That second one is what catches an edit to the **dataset card itself** — a changed licence paragraph, a changed
contact address — because the manifest covers only the three data files.

**0 means every check ran and passed. 2 means what ran passed and the
key was never checked against the anchor. 1 means something failed.**


In [ ]:
import subprocess, sys

# ../KEY-FINGERPRINT.txt was fetched from the ANCHOR two cells up,
# which is the whole point: the key travelled with the data and the
# fingerprint did not.
r = subprocess.run(
    [sys.executable, 'verify_sample.py', '.',
     '--fingerprint', '../KEY-FINGERPRINT.txt'],
    cwd='sample', capture_output=True, text=True)
print(r.stdout)
if r.stderr:
    print(r.stderr)
print('exit status:', r.returncode)
assert r.returncode == 0, (
    'the shipped verifier did not pass every check — read its output above before going further')


---
# Now the data

Everything above was proof. Everything below is the product.

## 5 · The six link states

These are not degrees of the same thing. Each is a different statement,
and the rare ones are the honest ones. Here is what each one means —
these definitions come from the code that assigns the states, not from
a copy kept beside this notebook:

- `not_an_amendment` — The filing is not an amendment — its form type carries no `/A` suffix. No link is expected and none is missing.
- `linked` — An amendment matched to the specific earlier filing it amends, held in this corpus, with the matching basis and confidence recorded on the row. This is the only state that asserts a link, and the only one that ever carries `amends_accession`.
- `unknown_subject` — No link was attempted, for one of two reasons, and `link_basis` says which. Either the form's chain is keyed on the SUBJECT of the filing — a Section 16 report names the insider it is about — and that subject is not in the index (`index:subject_unavailable`); or the filing it amends is identified only inside the document body, which this product does not open (`index:target_needs_document`). Either way the filing cannot be grouped with its own earlier versions from the index alone, so no link is attempted rather than one being guessed from the filer.
- `no_prior_filing_in_corpus` — It is an amendment, and the filing it amends is not in this corpus: either nothing else shares its chain, or it is the earliest member of one. This is a statement about OUR coverage, not about the filing — the original exists at the Commission.
- `unresolved_prior_exists` — An earlier filing in the chain exists but was WITHDRAWN, so the amendment has no live filing to point at. The prior is known to have been there and known not to be resolvable now, which is a different fact from never having found one.
- `unverifiable_at_release` — It would have linked, and the link is deliberately NOT asserted. The amendment's own document is not held at this release, and the accuracy audit confirms a link by reading that document — so no sampling and no census could ever check this row. Shipping it as `linked` would give it an implied verification it does not have, indistinguishable by a WHERE clause from rows that do. The target is dropped rather than carried with a caveat. Unreachable is not unread: no mode can reach it.

In [ ]:
import collections

rows = list(csv.DictReader(open('sample/hsh-amends-sample.csv',
                                newline='', encoding='utf-8')))
states = collections.Counter(json.loads(r['payload'])['link_state'] for r in rows)
for s, c in states.most_common():
    print(f'{s:28s} {c:>6,}')
print('\nNOTE: this is a STRATIFIED sample. Rare states are over-represented\n'
      'on purpose so you can see them. The full-corpus shares are in the\n'
      'README and in COVERAGE-ATTESTATION.json.')

## 6 · A real amendment chain

One filing that amends another, with the basis on which we linked them.

In [ ]:
linked = [r for r in rows if json.loads(r['payload'])['link_state'] == 'linked']
# pick one that carries a company name — some filings do not
r = next((x for x in linked if x['company_name'].strip()), linked[0])
p = json.loads(r['payload'])
print(f"company     : {r['company_name']}")
print(f"cik / cik10 : {r['cik']} / {r['cik10']}")
print(f"this filing : {r['accession']}  ({p['form_type']}, filed {r['filed_at']})")
print(f"it amends   : {p['amends_accession']}")
print(f"basis       : {p['link_basis']}")
print(f"confidence  : {p['link_confidence']}")
print(f"chain pos   : version_seq={p['version_seq']}  is_latest={p['is_latest']}")
print(f"\nrecord_uid  : {r['record_uid']}")
print('That uid is stable across releases. It is the column to join on.')

## 7 · A census stratum beside a sampled one

The distinction the attestation makes, visible in the data. A **census**
stratum was examined in full — no sampling, and therefore no confidence
interval, because quoting one would imply an inference nobody made.

In [ ]:
att = json.load(open('sample/COVERAGE-ATTESTATION.json'))
print(f"{'form':8s} {'population':>10s} {'precision':>10s}  claim type")
for s in att['strata']:
    if s['verdict'] != 'SHIP':
        continue
    iv = 'no interval — every member examined' if s['claim_type'] == 'census' \
         else f"[{100*s['interval'][0]:.2f}, {100*s['interval'][1]:.2f}]"
    print(f"{s['base_form']:8s} {s['population']:>10,} "
          f"{100*s['precision']:>9.2f}%  {s['claim_type']} — {iv}")

held = [s for s in att['strata'] if s['verdict'] != 'SHIP']
if held:
    print('\nAND WHAT WE WITHHELD — measured below the floor, not shipped as linked:')
    for s in held:
        print(f"  {s['link_basis']} / {s['base_form']}: {s['population']:,} links, "
              f"measured {100*s['precision']:.2f}% against a {100*s['floor']:.0f}% floor")

---
## What a sample cannot prove

Coverage. A subset establishes nothing about its superset, so the corpus-level
figures — populations, reachable frames, precision, claim types — are published
as `COVERAGE-ATTESTATION.json`, signed with the same key you verified in cell 3.

You can check our coverage claims without us handing over the data they describe.

---

Questions about the data, or about this notebook:

**info@healingsunhaven.com**